# In Class Activity April 14th, 2026 - Alexa Dandridge

In [18]:
# pip install optuna

### Importing libraries, preparing data, initial EDA

In [19]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [20]:
# importing data
adult = pd.read_csv('adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [21]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





From initial EDA, the dataset contains 48,842 rows and 15 different features. Some variables contain "?", which indicates there may be missing data. This makes me consider XGBoost for modeling, but I also am reminded that I will need to handle these values in preprocessing. Since the dataset also contains a lot of categorical variables, I want to be aware of encoding the variables before modeling. I also see there are imbalances in the data and some numerical features show skewness. There are also some dupplicates in the dataset.


### Data Preprocessing (minimal) and Baseline Model

In [22]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [23]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [24]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The baseline model has a mean F1 score of about 0.712. The scores are relatively consistent across the different folds, which shows that the model is not super sensitive to different training sets. Since the model is currenlt using default parapemeters, I would tune the hyperparameters. Then, I would potentially experiment with feature engineering/ seeing if I need to drop any variables, and then I would continue testing if my model performance improved. 

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [25]:
# Exploring different features of XGBoost and using stratified k-fold cross validation to evaluate the model performance with different hyperparameters, using GridSearchCV
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'scale_pos_weight': [1, 3],
    'subsample': [0.8, 1.0]
}
grid_search = GridSearchCV(estimator=XGBClassifier(enable_categorical=True, random_state=42), 
                           param_grid=param_grid, 
                           cv=skf, 
                           scoring='f1',
                           n_jobs=-1)
grid_search.fit(X, y)
print(f'Best hyperparameters: {grid_search.best_params_}')
print(f'Best F1 score from GridSearchCV: {grid_search.best_score_}')


Best hyperparameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'scale_pos_weight': 3, 'subsample': 1.0}
Best F1 score from GridSearchCV: 0.7184739191780467


The model that performed teh best had hyperparameters: learning rate: 0.1, max depth: 5, n estimators: 200, scale_pos_weight: 3, subsample: 1.0, and the best F1 score was 0.718, which is really not better than the default model.

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [26]:
# using GridSearchCV for tuning the preferred XGBoost model

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'scale_pos_weight': [1, 2, 3],
    'subsample': [0.8, 1.0]
}

xgb_model = XGBClassifier(enable_categorical=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best hyperparameters:", grid_search.best_params_)
print("Best cross-validated F1 score:", grid_search.best_score_)

# Training final model using best hyperparameters
best_xgb = grid_search.best_estimator_
best_xgb.fit(X_train, y_train)

# Making predictions on test set
y_pred = best_xgb.predict(X_test)

# Evaluating final model
print("Test F1 Score:", f1_score(y_test, y_pred))


Best hyperparameters: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 300, 'scale_pos_weight': 2, 'subsample': 1.0}
Best cross-validated F1 score: 0.7292766370824684
Test F1 Score: 0.7267350157728707


The new best F1 score is 0.72, which is an improvement!

### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [27]:
# tuning xgboost classifier with RandomizedSearchCV

param_dist = {
    'n_estimators': np.arange(100, 401, 50),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10),
    'scale_pos_weight': np.linspace(1.0, 5.0, num=9),
    'subsample': np.linspace(0.7, 1.0, num=4)
}

xgb_random = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42, enable_categorical=True),
    param_distributions=param_dist,
    n_iter=20,
    cv=skf,
    scoring='f1',
    random_state=42,
    n_jobs=1
)

xgb_random.fit(X_train, y_train)

print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')

# build final model using best parameters from RandomizedSearchCV
xgb_random_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    n_estimators=xgb_random.best_params_['n_estimators'],
    max_depth=xgb_random.best_params_['max_depth'],
    learning_rate=xgb_random.best_params_['learning_rate'],
    scale_pos_weight=xgb_random.best_params_['scale_pos_weight'],
    subsample=xgb_random.best_params_['subsample']
)

xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_random))
print("Test F1 Score:", f1_score(y_test, y_pred_random))
print(f'\nClassification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')

Best parameters from RandomizedSearchCV: {'subsample': np.float64(1.0), 'scale_pos_weight': np.float64(2.0), 'n_estimators': np.int64(100), 'max_depth': np.int64(7), 'learning_rate': np.float64(0.07444444444444444)}
Best F1 score from RandomizedSearchCV: 0.7274374195264932
Test Accuracy: 0.8586344559320299
Test F1 Score: 0.7287369868395207

Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.79      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.84      0.82      9769
weighted avg       0.87      0.86      0.86      9769



The model performed well with an F1 score of 0.72, which is very similar to the score obtained previously. 

In [28]:
# tuning with Optuna

def objective(trial):
    params = {
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 400, step=50),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'random_state': 42,
        'enable_categorical': True
    }

    xgb_optuna = XGBClassifier(**params)

    cv_scores = cross_val_score(xgb_optuna, X_train, y_train, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build final model with best parameters from Optuna
xgb_optuna_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    scale_pos_weight=study.best_params['scale_pos_weight'],
    max_depth=study.best_params['max_depth'],
    learning_rate=study.best_params['learning_rate'],
    n_estimators=study.best_params['n_estimators'],
    subsample=study.best_params['subsample']
)

xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_optuna))
print("Test F1 Score:", f1_score(y_test, y_pred_optuna))
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')

[I 2026-04-15 12:47:23,342] A new study created in memory with name: no-name-5570bd50-6a1a-4f2b-8d73-f7cb93313a93


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-15 12:47:27,027] Trial 0 finished with value: 0.7068177597848286 and parameters: {'scale_pos_weight': 2.7584212712948886, 'max_depth': 10, 'learning_rate': 0.13082198479092577, 'n_estimators': 200, 'subsample': 0.8682310613131392}. Best is trial 0 with value: 0.7068177597848286.
[I 2026-04-15 12:47:28,330] Trial 1 finished with value: 0.7162144739987134 and parameters: {'scale_pos_weight': 2.7780537822567024, 'max_depth': 4, 'learning_rate': 0.2978403578700514, 'n_estimators': 200, 'subsample': 0.8692162014209844}. Best is trial 1 with value: 0.7162144739987134.
[I 2026-04-15 12:47:32,939] Trial 2 finished with value: 0.6934931873109919 and parameters: {'scale_pos_weight': 4.27010673764592, 'max_depth': 10, 'learning_rate': 0.24357080136322287, 'n_estimators': 250, 'subsample': 0.9127076705090714}. Best is trial 1 with value: 0.7162144739987134.
[I 2026-04-15 12:47:36,278] Trial 3 finished with value: 0.7081948448377327 and parameters: {'scale_pos_weight': 3.672571381643673,

Exception ignored in: <function ResourceTracker.__del__ at 0x104619800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x103895800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102531800>
Traceback (most recent call last

The best model using Optuna shows an F1 core of 0.72, which is very similar to previous validatin methods. 

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


I explored three different methods for tuning: GridSearchCV, RandomizedSearchCV, and Optuna. Each method produced F1 scored of about 0.72, but GridSearchCV produced the overall best with a score fo 0.729. I prefer GridSearchCV rigth now because it is what I am most comfortable with, but I want to continue trying these other methods because I know there is value in them too. 